In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

BASE_DIR = Path.cwd().parent

df = pd.read_csv(
    BASE_DIR / "data" / "customer_support_tickets.csv"
)

df = df.dropna(
    subset=[
        "Ticket Subject",
        "Ticket Description",
        "Ticket Type"
    ]
).copy()

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


df["text"] = (
    df["Ticket Subject"]
    + " "
    + df["Ticket Description"]
)

df["clean_text"] = df["text"].apply(
    clean_text
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["Ticket Type"],
    test_size=0.2,
    random_state=42,
    stratify=df["Ticket Type"]
)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(
    X_train
)

X_test_tfidf = vectorizer.transform(
    X_test
)

In [ ]:
category_model = LogisticRegression(
    max_iter=2000
)

category_model.fit(
    X_train_tfidf,
    y_train
)

In [ ]:
predictions = category_model.predict(
    X_test_tfidf
)

accuracy = accuracy_score(
    y_test,
    predictions
)

precision, recall, f1, _ = (
    precision_recall_fscore_support(
        y_test,
        predictions,
        average="weighted"
    )
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
print(
    classification_report(
        y_test,
        predictions
    )
)

In [ ]:
matrix = confusion_matrix(
    y_test,
    predictions
)

plt.figure(figsize=(10, 7))

sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    xticklabels=category_model.classes_,
    yticklabels=category_model.classes_
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Category Classification Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
category_results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions
})

category_results.head(20)